# Course 07 lab — Acceptance criteria, invariants, and evidence

Northstar Mutual must decide what the Course 06 broker-response implementation has actually established. We will build a layered evidence portfolio while keeping measurement separate from release authority.

## Reproducibility and safety

This notebook is offline, deterministic, credential-free, and uses only synthetic training fixtures. It does not call a model, production service, or external API. Run it from this lesson directory with **Restart & Run All**.

In [ ]:
import importlib.util, json, sys
from pathlib import Path

lesson = Path.cwd()
spec = importlib.util.spec_from_file_location('course07_notebook_lab', lesson / 'lab.py')
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print('Loaded deterministic Course 07 harness')

## The verification chain

![Requirement-to-runtime evidence chain](assets/requirement-to-runtime-evidence.svg)

A requirement is normative intent. A criterion makes acceptance observable. A test or evaluator produces evidence for named revisions. An accountable policy turns selected evidence into a gate. Runtime signals then reveal whether the deployed control remains effective.

## Experiment 1 — Inspect the acceptance contract

The contract deliberately mixes positive, negative, boundary, failure, staleness, security, compatibility, and state criteria. Stable IDs preserve traceability; observable outcomes avoid binding the contract to private implementation methods.

In [ ]:
contract = lab.load_contract()
kinds = sorted({item['kind'] for item in contract['acceptance_criteria']})
summary = {'criteria': len(contract['acceptance_criteria']), 'invariants': len(contract['invariants']), 'kinds': kinds}
assert summary['criteria'] == 14 and not lab.validate_acceptance_contract(contract)
summary

## Experiment 2 — Execute observable examples

Examples make concrete decisions reviewable. They do not establish behavior over every possible input, so we retain the exact passed and total counts.

In [ ]:
acceptance = lab.run_acceptance_suite()
acceptance_summary = {'passed': sum(item.passed for item in acceptance), 'total': len(acceptance)}
assert acceptance_summary == {'passed': 14, 'total': 14}
[(item.criterion_id, item.passed, item.observations) for item in acceptance]

## Experiment 3 — Broaden the declared population

The bounded properties explore unauthorized inputs, conflicting value pairs, unrelated-field preservation, and stable response identities. Their finite populations make the result reproducible but do not constitute universal proof.

In [ ]:
properties = lab.property_suite()
assert len(properties) == 4 and all(item.violations == 0 for item in properties)
[(item.property_id, item.checked, item.violations) for item in properties]

## Experiment 4 — Separate structural coverage from correctness

Every declared decision-table row is exercised, yet the interpretation explicitly refuses to call the table correct. A complete map of the wrong policy is still wrong.

In [ ]:
coverage = lab.decision_table_coverage()
assert coverage['covered'] == coverage['total'] == 8
coverage

## Experiment 5 — Ask whether the evidence notices plausible faults

The harness seeds one requirement-elaboration mutation and two implementation mutations. Killing them demonstrates sensitivity to those faults—not completeness.

In [ ]:
mutations = lab.mutation_evidence()
assert (mutations['killed'], mutations['total']) == (3, 3)
mutations

## Experiment 6 — Define the AI evaluation before reading the score

The evaluation contract names the population, exclusions, protected split, label provenance, metrics, slices, and limitations. Predictions are fixed fixtures; no language model is evaluated here.

In [ ]:
assert not lab.evaluation_contract_findings()
baseline = lab.evaluation_metrics(prediction_key='baseline_prediction')
governed = lab.evaluation_metrics(prediction_key='governed_prediction')
assert baseline['overall']['value'] < governed['overall']['value']
{'baseline': baseline['overall'], 'governed': governed['overall'], 'slices': governed['slices'], 'limitations': governed['limitations']}

## Experiment 7 — Design human judgment without fabricating results

The rubric names reviewer qualification, blind independent ratings, behavioral anchors, adjudication, and agreement denominators. It remains `template_only_not_run`, so the course gains a review protocol without manufacturing human evidence.

In [ ]:
rubric = lab.load_json(lab.REFERENCE_ROOT / 'human-rubric.json')
assert rubric['status'] == 'template_only_not_run' and rubric['release_threshold'] is None
assert rubric['review_protocol']['reviewers_per_case'] >= 2 and rubric['review_protocol']['independent_before_adjudication']
{'status': rubric['status'], 'protocol': rubric['review_protocol'], 'dimensions': [item['id'] for item in rubric['dimensions']]}

## Experiment 8 — Validate the evidence bundle

Each record declares a producer relationship, validity tuple, result, denominator, timestamp, and limitations. Descriptive fixture identities are not authenticated attestations.

In [ ]:
records = lab.load_evidence_records()
assert len(records) == 13 and not lab.evidence_bundle_findings(records=records)
[(item['evidence_id'], item['evidence_class'], item['result']['status']) for item in records]

## Experiment 9 — Test freshness and invalidation

Evidence is valid only for its named specification, implementation, environment, dataset, and tool context. Changing the implementation revision makes prior records stale; the invalidation matrix identifies review scope rather than blindly ordering edits.

In [ ]:
manifest = lab.load_json(lab.EVIDENCE_MANIFEST_PATH)
current = lab.evidence_freshness(manifest, records)
changed = json.loads(json.dumps(manifest))
changed['current_context']['implementation_revision'] = 'course06-next'
stale = lab.evidence_freshness(changed, records)
assert all(item.current for item in current) and all(not item.current for item in stale)
{'current': sum(item.current for item in current), 'stale_after_change': sum(not item.current for item in stale), 'authorization_policy_scope': lab.invalidated_evidence('authorization_policy')}

## Experiment 10 — Keep measurement separate from authority

The deterministic criteria and safety-property gates have approved thresholds. The statistical conflict-recall threshold is intentionally unresolved, so the gate blocks even though a measurement exists.

In [ ]:
gates = lab.gate_results()
gate_view = {item.gate_id: {'decision': item.decision.value, 'reasons': item.reason_codes, 'numerator': item.numerator, 'denominator': item.denominator} for item in gates}
assert gate_view['GATE-BR-CONFLICT-EVAL']['decision'] == 'blocked'
gate_view

## Experiment 11 — Preserve the runtime denominator

Zero violations over five applicable synthetic events is measured. Zero violations over zero events is **not measured**; it must never be presented as a 0% production violation rate.

In [ ]:
events = lab.load_json(lab.RUNTIME_EVENTS_PATH)['events']
measured = lab.runtime_invariant_rate(events)
empty = lab.runtime_invariant_rate([])
assert measured['applicable_events'] == 5 and empty['status'] == 'not_measured' and empty['rate'] is None
{'measured': measured, 'empty_population': empty}

## Experiment 12 — Inspect the final claim-to-proof map

The summary carries both results and limitations. This is the minimum honest handoff: what was checked, over which population, against which revisions, and what remains unproven.

In [ ]:
claim_map = lab.claim_to_proof_map()
assert claim_map['acceptance'] == {'passed': 14, 'total': 14}
assert claim_map['runtime']['applicable_events'] == 5 and len(claim_map['limitations']) >= 4
claim_map

## Deliberate failure exercises

1. Run the acceptance suite with `unsafe_conflict=True` and identify the failed criterion.
2. Evaluate the Course 06 mutated decision table and compare row coverage with semantic failures.
3. Change the environment digest in the current context and inspect evidence invalidation.
4. Remove a denominator from one evidence record and run bundle validation.
5. Propose a statistical release threshold, owner, rationale, expiry, and rollback response—then have a peer challenge the authority and population.

## Production upgrade path

Replace training fixtures with representative, leakage-controlled datasets; calibrated human review; authenticated provenance and signed attestations; deployed contract environments; mutation campaigns tied to defect history; and runtime telemetry with privacy controls, trustworthy denominators, alert ownership, and rollback playbooks. Never promote this notebook's synthetic score into a production threshold.

## Workshop handoff

Open `northstar-broker-evidence/ticket/AI-2219-verification.md`, complete the files in `workshop/starter/`, and explain each missing claim, population, owner, provenance field, and stop condition before comparing with `reference/`.